In [1]:
import pandas as pd

from scipy import stats

In [2]:
df = pd.read_csv("./social_anxiety_dataset.csv")
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2030 entries, 0 to 2029
Data columns (total 22 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   Age                                1968 non-null   float64
 1   Gender                             1911 non-null   str    
 2   Occupation                         2030 non-null   str    
 3   Sleep Hours                        1994 non-null   float64
 4   Physical Activity (hrs/week)       2030 non-null   float64
 5   Caffeine Intake (mg/day)           1936 non-null   float64
 6   Alcohol Consumption (drinks/week)  2030 non-null   int64  
 7   Smoking                            1924 non-null   str    
 8   Family History of Anxiety          2030 non-null   str    
 9   Stress Level (1-10)                2030 non-null   int64  
 10  Heart Rate (bpm)                   2030 non-null   int64  
 11  Breathing Rate (breaths/min)       2030 non-null   int64  
 12  Swe

In [3]:
n_rows = df.shape[0]
print(f"dataset has {n_rows} data points")

dataset has 2030 data points


In [4]:
n_duplicates = (df.duplicated() == True).sum()
if n_duplicates == 0:
    print("dataset has no duplicate rows")
else:
    print(f"dataset has {n_duplicates} number of duplicate rows")

dataset has no duplicate rows


In [5]:
def print_general_metrics(df: pd.DataFrame, col: str):
    target = df[col]

    dtype = target.dtype

    n_rows = target.shape[0]
    n_missing = target.isna().sum()
    n_unique = target.nunique()

    print(f"--- Column `{col}` ---")
    print(f"    {"type":>15}: {dtype}")
    print(f"    {"count":>15}: {n_rows - n_missing}")
    print(f"    {"missing values":>15}: {n_missing} ({n_missing / n_rows * 100:4.2f}%)")
    print(f"    {"unique values":>15}: {n_unique}")
    print()


def print_numerical_metrics(df: pd.DataFrame, col: str):
    target = df[col]

    mean = target.mean()
    std = target.std(ddof=1)

    min_value = target.min()
    max_value = target.max()

    Q1, Q2, Q3 = target.quantile([0.25, 0.5, 0.75])

    lbound = Q1 - (Q3 - Q1) * 1.5
    ubound = Q3 + (Q3 - Q1) * 1.5

    n_outlier = ((target < lbound) | (target > ubound)).sum()
    n_zeros = (target == 0).sum()
    n_negatives = (target < 0).sum()

    print(f"--- Column `{col}` (numerical metrics) ---")
    print(f"    {"mean":>20}: {mean:0.2f}")
    print(f"    {"(trimmed) mean":>20}: {stats.trim_mean(target[~target.isna()], proportiontocut=0.1):0.2f}")
    print(f"    {"standard deviation":>20}: {std:0.2f}")
    print(f"    {"min":>20}: {min_value:0.2f}")
    print(f"    {"max":>20}: {max_value:0.2f}")
    print(f"    {"range":>20}: {max_value - min_value}")
    print(f"    {"mid-range":>20}: {(max_value - min_value) / 2}")
    print(f"    {"skewness":>20}: {target.skew():0.2f}")
    print(f"    {"Q1":>20}: {Q1}")
    print(f"    {"(median) Q2":>20}: {Q2}")
    print(f"    {"Q3":>20}: {Q3}")
    print(f"    {"(1.5x IQR) outliers":>20}: {n_outlier}")
    print(f"    {"anomalies":>20}: zeros: {n_zeros:02d} | negatives: {n_negatives:02d}")
    print()


def print_categorical_metrics(df: pd.DataFrame, col: str):
    target = df[col]

    n_rows = target.shape[0]

    value_counts = target.value_counts()
    mode_value = value_counts.index[0]
    mode_frequency = value_counts.iloc[0]

    print(f"--- Column `{col}` (categorical metrics) ---")
    print(f"    {"mode":>15}: {str(mode_value):s} ({mode_frequency / n_rows * 100:0.2f}%)")
    print()

    print(f"{'Categories:':>15}")
    for value, count in value_counts.items():
        print(f"{value:>15}: {count}")
    print()

In [6]:
def handle_missing_values(df: pd.DataFrame, col: str, method: str):
    if method not in ("mode", "mean", "median", "trimmed mean"):
        raise ValueError("method must be one of `mode`, `mean`, `trimmed mean` or `median`")

    method_function = {
        "mode": lambda s: s.mode()[0],
        "mean": lambda s: s.mean(),
        "median": lambda s: s.median(),
        "trimmed mean": lambda s: stats.trim_mean(s[~s.isna()], proportiontocut=0.1),
    }

    df[col] = df[col].fillna(method_function[method](df[col]))

In [7]:
numerical_columns = [
    "Age",
    "Sleep Hours",
    "Physical Activity (hrs/week)",
    "Caffeine Intake (mg/day)",
    "Alcohol Consumption (drinks/week)",
    "Heart Rate (bpm)",
    "Breathing Rate (breaths/min)",
    "Therapy Sessions (per month)",
]

categorical_columns = [
    "Gender",
    "Occupation",
    "Smoking",
    "Family History of Anxiety",
    "Stress Level (1-10)",
    "Sweating Level (1-5)",
    "Dizziness",
    "Medication",
    "Recent Major Life Event",
    "Diet Quality (1-10)",
    "Anxiety Level (1-10)",
    "Therapy History",
]

In [8]:
def analyze(
    df: pd.DataFrame,
    col: str,
    general_metrics: bool = True,
    numerical_metrics: bool = True,
    categorical_metrics: bool = True,
):
    if general_metrics:
        print_general_metrics(df, col)

    if numerical_metrics:
        print_numerical_metrics(df, col)

    if categorical_metrics:
        print_categorical_metrics(df, col)

In [9]:
droppables = []

# COLUMNS
- `Age`: individual's age in years.
- `Gender`: gender of the individual.
- `Occupation`: individual's job
- `Sleep Hours`: (average) number of hours the individual sleeps per night
- `Physical Activity`: total hours of exercise or physical activity the individual gets in a week
- `Caffeine Intake`: (estimated) daily consumption of caffeine measured in milligrams
- `Alcohol Consumption`: (average) number of alcoholic drink consumptions in a week
- `Smoking`: individual's smoking habit
- `Family History of Anxiety`: indicates whether the individual has blood relatives diagnosed with anxiety
- `Stress Level (1-10)`: an ordinal scale measuring the individual's perceived daily stress
- `Heart Rate`: (average) individual's measured heart rate in beats per minute
- `Breathing Rate`: individual's breathing rate measured in breaths per minute.
- `Sweating Level`: a scale measuring the intensity of the individual's sweating (often linked to physiological stress)
- `Dizziness`: indicates the presence dizziness
- `Medication`: indicates any current pharmacological treatments on the individual
- `Therapy Sessions`: number of mental health or counseling sessions the individual attends each month
- `Recent Major Life Event`: indicates if the individual has recently experienced a significant stressor
- `Diet Quality (1-10)`: an ordinal scale evaluating the overall health of the individual's diet
- `Anxiety Level (1-10)`: an ordinal scale measuring the severity of the individual's current anxiety
- `Target`: label for modeling
- `is_Anxious`: a binary flag classifying whether the individual is considered actively anxious (same as `Target`)
- `Therapy History`: information on the individual's past psychological treatment.

## `Age`

In [10]:
col = "Age"

analyze(df, col, categorical_metrics=False)

--- Column `Age` ---
               type: float64
              count: 1968
     missing values: 62 (3.05%)
      unique values: 47

--- Column `Age` (numerical metrics) ---
                    mean: 39.92
          (trimmed) mean: 39.71
      standard deviation: 13.24
                     min: 18.00
                     max: 64.00
                   range: 46.0
               mid-range: 23.0
                skewness: 0.11
                      Q1: 29.0
             (median) Q2: 40.0
                      Q3: 51.0
     (1.5x IQR) outliers: 0
               anomalies: zeros: 00 | negatives: 00



The `Age` column contains perfectly valid entries ranging from 18 to 64, with no zeros, negatives, or IQR-based outliers. Since the missing data rate is exceptionally low at 3.05%, those rows can be safely dropped without impacting the dataset.

In [11]:
# drop age missing values
droppables.append("Age")

## `Gender`

In [12]:
col = "Gender"

analyze(df, col, numerical_metrics=False)

--- Column `Gender` ---
               type: str
              count: 1911
     missing values: 119 (5.86%)
      unique values: 3

--- Column `Gender` (categorical metrics) ---
               mode: Female (32.36%)

    Categories:
         Female: 657
          Other: 627
           Male: 627



The `Gender` column has three well-balanced categories, with `Female` as the mode at only 32.36% — no category dominates enough to safely impute from. Rather than dropping the 5.86% missing rows (which would still discard real signal), a new `Unkown` category is added to explicitly flag missing responses instead of forcing them into an existing group.

In [13]:
df.loc[df["Gender"].isna(), "Gender"] = "Unknown"

## `Occupation`

In [14]:
col = "Occupation"

analyze(df, col, numerical_metrics=False)

--- Column `Occupation` ---
               type: str
              count: 2030
     missing values: 0 (0.00%)
      unique values: 13

--- Column `Occupation` (categorical metrics) ---
               mode: Student (8.97%)

    Categories:
        Student: 182
         Artist: 172
         Lawyer: 167
       Musician: 163
          Nurse: 162
         Doctor: 161
      Scientist: 155
        Teacher: 151
          Other: 149
        Athlete: 147
           Chef: 146
       Engineer: 141
     Freelancer: 134



The `Occupation` column has no missing values and 13 fairly evenly distributed categories, with `Student` as the mode at only 8.97%.

## `Sleep Hours`

In [15]:
col = "Sleep Hours"

analyze(df, col, categorical_metrics=False)

--- Column `Sleep Hours` ---
               type: float64
              count: 1994
     missing values: 36 (1.77%)
      unique values: 76

--- Column `Sleep Hours` (numerical metrics) ---
                    mean: 6.42
          (trimmed) mean: 6.66
      standard deviation: 2.16
                     min: -10.00
                     max: 11.00
                   range: 21.0
               mid-range: 10.5
                skewness: -4.25
                      Q1: 5.8
             (median) Q2: 6.7
                      Q3: 7.5
     (1.5x IQR) outliers: 52
               anomalies: zeros: 00 | negatives: 40



The `Sleep Hours` column has 52 IQR-based outliers, including the 40 physically impossible negative entries, plus 36 separately missing values (1.77%). 

Rather than dropping the outlier rows, they are flagged in a new `Sleep Hours Flag` column and then capped to the IQR lower/upper bounds, preserving the rows while limiting their influence; the genuinely missing rows are still dropped.

In [16]:
df["Sleep Hours Flag"] = 0

Q1 = df["Sleep Hours"].quantile(0.25)
Q3 = df["Sleep Hours"].quantile(0.75)

lower_bound = Q1 - 1.5 * (Q3 - Q1)
upper_bound = Q3 + 1.5 * (Q3 - Q1)

df.loc[(df["Sleep Hours"] < lower_bound) | (df["Sleep Hours"] > upper_bound), "Sleep Hours Flag"] = 1

In [17]:
droppables.append("Sleep Hours")

## `Physical Activity (hrs/week)`

In [18]:
col = "Physical Activity (hrs/week)"

analyze(df, col, categorical_metrics=False)

--- Column `Physical Activity (hrs/week)` ---
               type: float64
              count: 2030
     missing values: 0 (0.00%)
      unique values: 89

--- Column `Physical Activity (hrs/week)` (numerical metrics) ---
                    mean: 2.80
          (trimmed) mean: 2.83
      standard deviation: 2.23
                     min: -10.00
                     max: 9.20
                   range: 19.2
               mid-range: 9.6
                skewness: -1.44
                      Q1: 1.4
             (median) Q2: 2.8
                      Q3: 4.2
     (1.5x IQR) outliers: 33
               anomalies: zeros: 14 | negatives: 40



The `Physical Activity (hrs/week)` column has no missing values, but 40 rows with impossible values, amounting to 1.97% of the data.

In [19]:
df.loc[df["Physical Activity (hrs/week)"] < 0, "Physical Activity (hrs/week)"] = float("nan")
droppables.append("Physical Activity (hrs/week)")

## `Caffeine Intake (mg/day)`

In [20]:
col = "Caffeine Intake (mg/day)"

analyze(df, col, categorical_metrics=False)

--- Column `Caffeine Intake (mg/day)` ---
               type: float64
              count: 1936
     missing values: 94 (4.63%)
      unique values: 544

--- Column `Caffeine Intake (mg/day)` (numerical metrics) ---
                    mean: 306.28
          (trimmed) mean: 285.84
      standard deviation: 208.01
                     min: 0.00
                     max: 1500.00
                   range: 1500.0
               mid-range: 750.0
                skewness: 2.91
                      Q1: 177.75
             (median) Q2: 277.5
                      Q3: 391.0
     (1.5x IQR) outliers: 30
               anomalies: zeros: 02 | negatives: 00



The `Caffeine Intake (mg/day)` column ranges from 0 to 1500mg with a strongly right-skewed distribution and 30 IQR-based outliers on the high end. Since data is skew, the mean is pulled well above the median, the median is used instead, and rows are filled rather than dropped since the missing rate isn't low enough to discard without meaningful data loss.

In [21]:
handle_missing_values(df, "Caffeine Intake (mg/day)", "median")

## `Alcohol Consumption (drinks/week)`

In [22]:
col = "Alcohol Consumption (drinks/week)"

analyze(df, col, categorical_metrics=False)

--- Column `Alcohol Consumption (drinks/week)` ---
               type: int64
              count: 2030
     missing values: 0 (0.00%)
      unique values: 23

--- Column `Alcohol Consumption (drinks/week)` (numerical metrics) ---
                    mean: 9.49
          (trimmed) mean: 9.64
      standard deviation: 6.03
                     min: -10.00
                     max: 19.00
                   range: 29
               mid-range: 14.5
                skewness: -0.26
                      Q1: 5.0
             (median) Q2: 10.0
                      Q3: 15.0
     (1.5x IQR) outliers: 0
               anomalies: zeros: 95 | negatives: 40



The `Alcohol Consumption (drinks/week)` column has no missing values, but has 40 rows with impossible values. These negatives will still be treated as invalid and dropped.

In [23]:
df.loc[df["Alcohol Consumption (drinks/week)"] < 0, "Alcohol Consumption (drinks/week)"] = float("nan")
droppables.append("Alcohol Consumption (drinks/week)")

## `Smoking`

In [24]:
col = "Smoking"

analyze(df, col, numerical_metrics=False)

--- Column `Smoking` ---
               type: str
              count: 1924
     missing values: 106 (5.22%)
      unique values: 2

--- Column `Smoking` (categorical metrics) ---
               mode: Yes (50.25%)

    Categories:
            Yes: 1020
             No: 904



The `Smoking` column is a near-even split with 5.22% missing values. A new `Unknown` category is added to explicitly flag missing responses rather than forcing them into an existing group.

In [25]:
df.loc[df["Smoking"].isna(), "Smoking"] = "Unknown"

## `Family History of Anxiety`

In [26]:
col = "Family History of Anxiety"

analyze(df, col, numerical_metrics=False)

--- Column `Family History of Anxiety` ---
               type: str
              count: 2030
     missing values: 0 (0.00%)
      unique values: 2

--- Column `Family History of Anxiety` (categorical metrics) ---
               mode: Yes (52.12%)

    Categories:
            Yes: 1058
             No: 972



The `Family History of Anxiety` column has no missing values and a roughly even split.

## `Stress Level (1-10)`

In [27]:
col = "Stress Level (1-10)"

analyze(df, col)

--- Column `Stress Level (1-10)` ---
               type: int64
              count: 2030
     missing values: 0 (0.00%)
      unique values: 11

--- Column `Stress Level (1-10)` (numerical metrics) ---
                    mean: 6.03
          (trimmed) mean: 6.06
      standard deviation: 3.14
                     min: 1.00
                     max: 15.00
                   range: 14
               mid-range: 7.0
                skewness: 0.08
                      Q1: 3.0
             (median) Q2: 6.0
                      Q3: 9.0
     (1.5x IQR) outliers: 0
               anomalies: zeros: 00 | negatives: 00

--- Column `Stress Level (1-10)` (categorical metrics) ---
               mode: 10 (13.00%)

    Categories:
             10: 264
              9: 249
              8: 245
              1: 185
              6: 181
              3: 180
              7: 177
              2: 177
              5: 175
              4: 167
             15: 30



The `Stress Level (1-10)` column has no missing values and a symmetric distribution evenly spread across 1-10. However, 30 rows (1.48%) hold a value of 15, outside the column's stated 1-10 scale. Rather than dropping them, these rows are flagged in a new `High Stress Flag` column and the value is capped to 10, preserving what is likely a genuine high-stress signal instead of discarding it.

In [28]:
df["High Stress Flag"] = 0
df.loc[df["Stress Level (1-10)"] > 10, "High Stress Flag"] = 1
df["Stress Level (1-10)"] = df["Stress Level (1-10)"].clip(upper=10)

## `Heart Rate (bpm)`

In [29]:
col = "Heart Rate (bpm)"

analyze(df, col, categorical_metrics=False)

--- Column `Heart Rate (bpm)` ---
               type: int64
              count: 2030
     missing values: 0 (0.00%)
      unique values: 61

--- Column `Heart Rate (bpm)` (numerical metrics) ---
                    mean: 93.07
          (trimmed) mean: 91.92
      standard deviation: 23.31
                     min: 60.00
                     max: 220.00
                   range: 160
               mid-range: 80.0
                skewness: 2.20
                      Q1: 76.0
             (median) Q2: 93.0
                      Q3: 107.0
     (1.5x IQR) outliers: 30
               anomalies: zeros: 00 | negatives: 00



The `Heart Rate (bpm)` column has no missing values. It is right-skewed with 30 IQR-based outliers reaching up to 220 bpm — physiologically extreme but plausible for individuals experiencing acute anxiety/panic symptoms. Rather than dropping them, these rows are flagged in a new `Heart Rate Flag` column and then capped to the IQR lower/upper bounds, preserving the rows while limiting their influence.

In [30]:
df["Heart Rate Flag"] = 0

Q1 = df["Heart Rate (bpm)"].quantile(0.25)
Q3 = df["Heart Rate (bpm)"].quantile(0.75)

lower_bound = Q1 - 1.5 * (Q3 - Q1)
upper_bound = Q3 + 1.5 * (Q3 - Q1)

df.loc[(df["Heart Rate (bpm)"] < lower_bound) | (df["Heart Rate (bpm)"] > upper_bound), "Heart Rate Flag"] = 1
df["Heart Rate (bpm)"] = df["Heart Rate (bpm)"].clip(lower=lower_bound, upper=upper_bound)

## `Breathing Rate (breaths/min)`

In [31]:
col = "Breathing Rate (breaths/min)"

analyze(df, col, categorical_metrics=False)

--- Column `Breathing Rate (breaths/min)` ---
               type: int64
              count: 2030
     missing values: 0 (0.00%)
      unique values: 18

--- Column `Breathing Rate (breaths/min)` (numerical metrics) ---
                    mean: 20.96
          (trimmed) mean: 21.05
      standard deviation: 5.18
                     min: 12.00
                     max: 29.00
                   range: 17
               mid-range: 8.5
                skewness: -0.13
                      Q1: 17.0
             (median) Q2: 21.0
                      Q3: 26.0
     (1.5x IQR) outliers: 0
               anomalies: zeros: 00 | negatives: 00



The `Breathing Rate (breaths/min)` column has no missing values, a roughly symmetric distribution, and no IQR-based outliers.

## `Sweating Level (1-5)`

In [32]:
col = "Sweating Level (1-5)"

analyze(df, col)

--- Column `Sweating Level (1-5)` ---
               type: int64
              count: 2030
     missing values: 0 (0.00%)
      unique values: 5

--- Column `Sweating Level (1-5)` (numerical metrics) ---
                    mean: 3.10
          (trimmed) mean: 3.12
      standard deviation: 1.39
                     min: 1.00
                     max: 5.00
                   range: 4
               mid-range: 2.0
                skewness: -0.10
                      Q1: 2.0
             (median) Q2: 3.0
                      Q3: 4.0
     (1.5x IQR) outliers: 0
               anomalies: zeros: 00 | negatives: 00

--- Column `Sweating Level (1-5)` (categorical metrics) ---
               mode: 4 (21.48%)

    Categories:
              4: 436
              3: 434
              5: 426
              2: 374
              1: 360



The `Sweating Level (1-5)` column has no missing values and a near-symmetric distribution (skew=-0.10). The category breakdown confirms all values fall cleanly within the stated 1-5 scale, with `4` as a mild mode (21.48%).

## `Dizziness`

In [33]:
col = "Dizziness"

analyze(df, col, numerical_metrics=False)

--- Column `Dizziness` ---
               type: str
              count: 2030
     missing values: 0 (0.00%)
      unique values: 2

--- Column `Dizziness` (categorical metrics) ---
               mode: Yes (51.67%)

    Categories:
            Yes: 1049
             No: 981



The `Dizziness` column has no missing values and a roughly even split.

## `Medication`

In [34]:
col = "Medication"

analyze(df, col, numerical_metrics=False)

--- Column `Medication` ---
               type: str
              count: 1921
     missing values: 109 (5.37%)
      unique values: 2

--- Column `Medication` (categorical metrics) ---
               mode: Yes (47.49%)

    Categories:
            Yes: 964
             No: 957



The `Medication` column is an almost perfectly even split with 5.37% missing values. Since neither category dominates, imputing with the mode would arbitrarily bias 109 rows toward one class; instead, a new `Unknown` category is added to explicitly flag missing responses rather than forcing them into an existing group.

In [35]:
df.loc[df["Medication"].isna(), "Medication"] = "Unknown"

## `Therapy Sessions (per month)`

In [36]:
col = "Therapy Sessions (per month)"

analyze(df, col, categorical_metrics=False)

--- Column `Therapy Sessions (per month)` ---
               type: int64
              count: 2030
     missing values: 0 (0.00%)
      unique values: 11

--- Column `Therapy Sessions (per month)` (numerical metrics) ---
                    mean: 2.36
          (trimmed) mean: 2.07
      standard deviation: 2.15
                     min: 0.00
                     max: 10.00
                   range: 10
               mid-range: 5.0
                skewness: 1.05
                      Q1: 1.0
             (median) Q2: 2.0
                      Q3: 3.0
     (1.5x IQR) outliers: 119
               anomalies: zeros: 417 | negatives: 00



The `Therapy Sessions (per month)` column has no missing values. It is right-skewed with 417 legitimate zero entries and 119 IQR-based outliers on the high end, which are plausible for individuals in intensive treatment. Rather than dropping them, these rows are flagged in a new `Therapy Sessions Flag` column and then capped to the IQR lower/upper bounds, preserving the rows while limiting their influence.

In [37]:
df["Therapy Sessions Flag"] = 0

Q1 = df["Therapy Sessions (per month)"].quantile(0.25)
Q3 = df["Therapy Sessions (per month)"].quantile(0.75)

lower_bound = Q1 - 1.5 * (Q3 - Q1)
upper_bound = Q3 + 1.5 * (Q3 - Q1)

df.loc[(df["Therapy Sessions (per month)"] < lower_bound) | (df["Therapy Sessions (per month)"] > upper_bound), "Therapy Sessions Flag"] = 1
df["Therapy Sessions (per month)"] = df["Therapy Sessions (per month)"].clip(lower=lower_bound, upper=upper_bound)

## `Recent Major Life Event`

In [38]:
col = "Recent Major Life Event"

analyze(df, col, numerical_metrics=False)

--- Column `Recent Major Life Event` ---
               type: str
              count: 2030
     missing values: 0 (0.00%)
      unique values: 2

--- Column `Recent Major Life Event` (categorical metrics) ---
               mode: Yes (51.23%)

    Categories:
            Yes: 1040
             No: 990



## `Diet Quality (1-10)`

In [39]:
col = "Diet Quality (1-10)"

analyze(df, col)

--- Column `Diet Quality (1-10)` ---
               type: float64
              count: 1952
     missing values: 78 (3.84%)
      unique values: 10

--- Column `Diet Quality (1-10)` (numerical metrics) ---
                    mean: 5.23
          (trimmed) mean: 5.18
      standard deviation: 2.88
                     min: 1.00
                     max: 10.00
                   range: 9.0
               mid-range: 4.5
                skewness: 0.12
                      Q1: 3.0
             (median) Q2: 5.0
                      Q3: 8.0
     (1.5x IQR) outliers: 0
               anomalies: zeros: 00 | negatives: 00

--- Column `Diet Quality (1-10)` (categorical metrics) ---
               mode: 3.0 (11.03%)

    Categories:
            3.0: 224
            2.0: 223
            1.0: 221
            4.0: 210
            6.0: 190
            8.0: 185
            7.0: 179
            9.0: 175
           10.0: 173
            5.0: 172



The `Diet Quality (1-10)` column has a symmetric distribution, no IQR-based outliers, and the category breakdown confirms all values fall cleanly within the stated 1-10 scale. With 3.84% missing data, those rows can be safely dropped.

In [40]:
droppables.append("Diet Quality (1-10)")

## `Anxiety Level (1-10)`

In [41]:
col = "Anxiety Level (1-10)"

analyze(df, col)

--- Column `Anxiety Level (1-10)` ---
               type: float64
              count: 2030
     missing values: 0 (0.00%)
      unique values: 10

--- Column `Anxiety Level (1-10)` (numerical metrics) ---
                    mean: 3.92
          (trimmed) mean: 3.68
      standard deviation: 2.14
                     min: 1.00
                     max: 10.00
                   range: 9.0
               mid-range: 4.5
                skewness: 1.00
                      Q1: 2.0
             (median) Q2: 4.0
                      Q3: 5.0
     (1.5x IQR) outliers: 60
               anomalies: zeros: 00 | negatives: 00

--- Column `Anxiety Level (1-10)` (categorical metrics) ---
               mode: 3.0 (21.18%)

    Categories:
            3.0: 430
            4.0: 424
            2.0: 329
            5.0: 303
            1.0: 206
            6.0: 128
            8.0: 68
           10.0: 60
            9.0: 56
            7.0: 26



The `Anxiety Level (1-10)` column has no missing values, and the category breakdown confirms all values fall cleanly within the stated 1-10 scale.

## `Target`

In [42]:
col = "Target"

analyze(df, col, numerical_metrics=False)

--- Column `Target` ---
               type: int64
              count: 2030
     missing values: 0 (0.00%)
      unique values: 2

--- Column `Target` (categorical metrics) ---
               mode: 0 (89.66%)

    Categories:
              0: 1820
              1: 210



The `Target` column has no missing values and is a binary label, reflecting real-world anxiety.

## `is_Anxious`

In [43]:
col = "is_Anxious"

analyze(df, col, numerical_metrics=False)

--- Column `is_Anxious` ---
               type: int64
              count: 2030
     missing values: 0 (0.00%)
      unique values: 2

--- Column `is_Anxious` (categorical metrics) ---
               mode: 0 (89.66%)

    Categories:
              0: 1820
              1: 210



The `is_Anxious` column has no missing values and is identical to `Target` for every row. Since it's a fully redundant duplicate of the modeling label, it will be dropped rather than kept as a duplicate feature.

In [44]:
df = df.drop(columns=["is_Anxious"])

## `Therapy History`

In [45]:
col = "Therapy History"

analyze(df, col, numerical_metrics=False)

--- Column `Therapy History` ---
               type: str
              count: 203
     missing values: 1827 (90.00%)
      unique values: 4

--- Column `Therapy History` (categorical metrics) ---
               mode: No previous history (2.96%)

    Categories:
No previous history: 60
  Group Therapy: 54
Cognitive Behavioral Therapy (CBT): 45
Psychodynamic Therapy: 44



The `Therapy History` column is missing for 90.00% of records. Since the column is effectively unusable, it will be dropped entirely rather than the individual rows.

In [46]:
df = df.drop(columns=["Therapy History"])

In [47]:
df = df.dropna(subset=droppables)
df.info()

<class 'pandas.DataFrame'>
Index: 1784 entries, 0 to 2029
Data columns (total 24 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   Age                                1784 non-null   float64
 1   Gender                             1784 non-null   str    
 2   Occupation                         1784 non-null   str    
 3   Sleep Hours                        1784 non-null   float64
 4   Physical Activity (hrs/week)       1784 non-null   float64
 5   Caffeine Intake (mg/day)           1784 non-null   float64
 6   Alcohol Consumption (drinks/week)  1784 non-null   float64
 7   Smoking                            1784 non-null   str    
 8   Family History of Anxiety          1784 non-null   str    
 9   Stress Level (1-10)                1784 non-null   int64  
 10  Heart Rate (bpm)                   1784 non-null   float64
 11  Breathing Rate (breaths/min)       1784 non-null   int64  
 12  Sweating